In [ ]:
import { logError } from "./logError.js";
import { reloadEnv } from "./reloadEnv.js";
await reloadEnv();
const token = Deno.env.get("OST_TOKEN");
const daToken = Deno.env.get("DA_TOKEN");
const aosEndPoint = 'https://aos.adobe.io';
const wcsEndPoint = 'https://www.adobe.com';    
const api_key = 'wcms-commerce-ims-user-prod';
const environment = 'PROD'; 
const landscape = 'DRAFT'; // 'PUBLISHED';

const daEndPoint = 'https://admin.da.live'
const org = 'adobecom'
const site = 'upp'

const planTypesToTest = ['M2M', 'ABM', 'PUF'];
const countriesToTest = ['US', 'JP'];

const planTypes = {
  M2M: {
    commitment: 'MONTH',
    term: 'MONTHLY',
  },
  ABM: {
    commitment: 'YEAR',
    term: 'MONTHLY',
  },
  PUF: {
    commitment: 'YEAR',
    term: 'ANNUAL',
  }
};

In [ ]:
// Read a CSV file
const csv = await Deno.readTextFile('offers.csv');
const rows = csv.split('\n');
const headers = rows[0].split(',');
const data = rows.slice(1).map(row => row.split(','));
const offerData = data.map(row => {
  const offer = {};
  headers.forEach((header, index) => {
    offer[header] = row[index];
  });
  return offer;
});
console.log(offerData);


In [ ]:
// loop through offerData and call the AOS API for each offer
for (const offer of offerData) {
  const offerId = offer.OFFER_ID;
  const params = {
    environment,
    landscape,
    api_key,
  };
  const queryString = new URLSearchParams(params).toString();
  const url = new URL(`offers/${offerId}?${queryString}`, aosEndPoint).toString();
  const response = await fetch(url, {
    headers: {
      'X-Api-Key': api_key
    }
  });
  
  if (!response.ok) {
    await logError(response);
    continue;
  } 

  offer.details = await response.json();
  if (offer.details.length > 1) {
    console.log(`${offer.OFFER_ID} has multiple offers`);
  } else if (offer.details.length == 0) {
    console.log(`${offer.OFFER_ID} has no offers`);
  } else {
    offer.details = offer.details[0];
    console.log(JSON.stringify(offer.details, null, 2));
  }
}


In [ ]:
// loop through offerData and call the AOS API for each offer to get offer selector ID
for (const entry of offerData) {
  for (const planType of planTypesToTest) {
    const offer = {...entry.details, ...planTypes[planType]};
    entry[planType] = {};
    
    const params = {
      api_key
    }
    const body = {
      buying_program: offer.buying_program,
      commitment: offer.commitment,
      customer_segment: offer.customer_segment,
      market_segment: offer.market_segments[0],
      merchant: offer.merchant,
      offer_type: offer.offer_type,
      price_point: offer.price_point,
      product_arrangement_code: offer.product_arrangement_code,
      sales_channel: offer.sales_channel,
      term: offer.term,
    }

    const queryString = new URLSearchParams(params).toString();
    const url = new URL(`offer_selectors?${queryString}`, aosEndPoint).toString();
    
    const response = await fetch(url, {
      method: 'POST',
      headers: {
        'X-Api-Key': api_key,
        Authorization: `Bearer ${token}`,
        'Content-Type': 'application/json',
      },
      body: JSON.stringify(body),
    });
    
    if (!response.ok) {
      await logError(response);
      console.log('Error getting offer selector ID');
    }
    
    entry[planType].offerSelector = await response.json();
    console.log(entry[planType].offerSelector);
  }
}

In [ ]:
const template = `        
  <div class="merch-card product">
    <div>
      <div>
        <h3>{{product_arrangement_code}}</h3>
        <h3>{{customer_segment}} – {{commitment}} - {{term}} ({{planType}})<br><a href="https://milo.adobe.com/tools/ost?osi={{SelectorId}}&#x26;type=price">PRICE - {{term}} - {{product_code}}</a></h3>
        <p>Offer ID: {{offerId}}</p>
        <p><a href="https://milo.adobe.com/tools/ost?osi={{SelectorId}}&#x26;type=checkoutUrl&#x26;text=buy-now&#x26;workflowStep=commitment">CTA {{buy-now}}</a></p>
      </div>
    </div>
  </div>`;

const pages = {};

for (const entry of offerData) {  
  let countries = countriesToTest;
  if (countries === 'ALL') {
    countries = offer.details.countries;
  }

  // loop through offer.countries and call the WCS API for each country
  for (const country of countries) {
    if (!pages[country]) {
      pages[country] = '';
    }
    let mainHtml = pages[country];
    for (const planType of planTypesToTest) {
      const offer = {...entry.details, ...planTypes[planType]};
      const offerSelector = entry[planType].offerSelector;
      
      const productHtml = template
        .replaceAll('{{product_arrangement_code}}', offer.product_arrangement_code)
        .replaceAll('{{customer_segment}}', offer.customer_segment)
        .replaceAll('{{term}}', offer.term)
        .replaceAll('{{commitment}}', offer.commitment)
        .replaceAll('{{offerId}}', offer.offer_id)
        .replaceAll('{{SelectorId}}', offerSelector.id)
        .replaceAll('{{planType}}', planType)
      mainHtml += productHtml;
    }
    pages[country] = mainHtml;
  }
}

for (const country of Object.keys(pages)) {
  pages[country] = `<!DOCTYPE html><html><head><title>Offer Test Page<\/title><\/head><body><main><div>${pages[country]}<\/div><\/main><\/body><\/html>`;
}

for (const country of Object.keys(pages)) {
  console.log(pages[country]);
}

In [ ]:
const pagePath = '/drafts/tsay/offer-test.html';
const basename = pagePath.split('/').pop();

for (const country of Object.keys(pages)) {
  console.log(`Write the test page for ${country}`);

  let locale = country.toLowerCase();
  locale = locale === 'us' ? '' : `/${locale}`;

  const url = new URL(`source/${org}/${site}${locale}${pagePath}`, daEndPoint).toString();

  const formData = new FormData();

  const fileBlob = new Blob([pages[country]]);
  formData.append('data', fileBlob, basename);

  const response = await fetch(url, {
    method: 'POST',
    headers: {
      Authorization: `Bearer ${daToken}`,
    },
    body: formData
  });

  if (!response.ok) {
    await logError(response);
    throw new Error('Request failed!');
  }

  const newPage = await response.json();
  console.log(newPage);
}